# Complete End-to-End RAG Pipeline
### PDF Cleaning $\rightarrow$ Semantic Chunking $\rightarrow$ Local Vector Database $\rightarrow$ Advanced Retrievers & LCEL

This project demonstrates a production-grade RAG system for **`Alderbridge-Employee-Rulebook.pdf`**:
1. **Data Ingestion & Cleaning:** PyPDF extraction with regex-based removal of running headers/footers, hyphenation healing, and Markdown formatting.
2. **Semantic Chunking:** Context-aware semantic text splitting based on embedding distance shifts.
3. **Local Persistent Vector Database:** Stored locally in **Chroma** using normalized `sentence-transformers/all-MiniLM-L6-v2` embeddings and Cosine distance.
4. **Configurable Retrievers:** Standard Similarity Retriever, Maximal Marginal Relevance (MMR) Retriever, and Score-Threshold Retriever.
5. **LCEL RAG Chain:** Prompt engineering and context synthesis ready for LLM generation.


In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-experimental langchain-chroma chromadb langchain-huggingface sentence-transformers pypdf langchain-google-genai


### 1. Import Required Libraries


In [1]:
import os
import re
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


### 2. Load PDF Document
Loads `Alderbridge-Employee-Rulebook.pdf` and verifies that the file exists locally.


In [2]:
pdf_path = "Alderbridge-Employee-Rulebook.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found at: {pdf_path}")

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Total pages loaded: {len(documents)}")
print("\n--- Raw Text Preview (Page 3 before cleaning) ---")
print(repr(documents[2].page_content[:250]))


Total pages loaded: 8

--- Raw Text Preview (Page 3 before cleaning) ---
'ALDERBRIDGE OFFICE SYSTEMS\nEMPLOYEE RULEBOOK  /  VERSION 1.0\nSECTION 01\nEmployment and onboarding\n01.01\nScope and policy ownership\nAlderbridge Office Systems is a fictional provider of workplace equipment and support services. This\nhandbook describes'


### 3. Data Cleaning Pipeline
Removes document noise that harms retrieval precision:
- Strips running headers (`ALDERBRIDGE OFFICE SYSTEMS / EMPLOYEE RULEBOOK`)
- Strips footers and page counts (`FICTIONAL COMPANY / Page X of Y`)
- Heals hyphenated line splits (`docu-\nment` $\rightarrow$ `document`)
- Joins soft line wraps into fluid sentences while preserving Markdown section/policy headings.


In [3]:
def clean_pdf_text(text: str) -> str:
    """
    Cleans raw PDF extracted text:
    1. Removes repetitive headers and footers/page numbers.
    2. Reconnects hyphenated line-broken words.
    3. Formats Section and Policy numbers into structured markdown headings.
    4. Merges soft line wraps within sentences.
    5. Normalizes whitespace and redundant newlines.
    """
    # 1. Remove running headers
    text = re.sub(
        r'(?i)ALDERBRIDGE\s+OFFICE\s+SYSTEMS\s*\n\s*(?:EMPLOYEE\s+RULEBOOK\s*/\s*VERSION\s*[\d\.]+\s*\n)?',
        '',
        text
    )
    
    # 2. Remove running footers and page numbering
    text = re.sub(r'(?i)FICTIONAL\s+COMPANY\s*/\s*EMPLOYEE\s+RULEBOOK\s*\n\s*Page\s*\d+\s*of\s*\d+', '', text)
    text = re.sub(r'(?i)^\s*Page\s*\d+\s*of\s*\d+\s*$', '', text, flags=re.MULTILINE)
    
    # 3. Heal hyphenated line breaks
    text = re.sub(r'(\b\w+)-\s*\n\s*(\w+\b)', r'\1\2', text)
    
    # 4. Standardize Section and Policy numbers into structured markdown headings
    text = re.sub(r'^\s*SECTION\s+(\d+)\s*\n\s*([^\n]+)', r'\n### SECTION \1: \2\n', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*(\d{2}\.\d{2})\s*\n\s*([^\n]+)', r'\n#### Policy \1: \2\n', text, flags=re.MULTILINE)

    # 5. Join soft line breaks inside paragraphs
    lines = text.split('\n')
    cleaned_lines = []
    current_para = []
    
    for line in lines:
        stripped = line.strip()
        if not stripped:
            if current_para:
                cleaned_lines.append(' '.join(current_para))
                current_para = []
            cleaned_lines.append('')
        elif stripped.startswith('#') or stripped.startswith('*') or stripped.startswith('-'):
            if current_para:
                cleaned_lines.append(' '.join(current_para))
                current_para = []
            cleaned_lines.append(stripped)
        else:
            current_para.append(stripped)
            
    if current_para:
        cleaned_lines.append(' '.join(current_para))
        
    result = '\n'.join(cleaned_lines)
    result = re.sub(r'\n{3,}', '\n\n', result)
    result = re.sub(r'[ \t]+', ' ', result)
    return result.strip()

# Apply cleaning across all document pages
for doc in documents:
    doc.page_content = clean_pdf_text(doc.page_content)

print("--- Cleaned Text Preview (Page 3 after cleaning) ---")
print(documents[2].page_content[:380])


--- Cleaned Text Preview (Page 3 after cleaning) ---
### SECTION 01: Employment and onboarding

#### Policy 01.01: Scope and policy ownership

Alderbridge Office Systems is a fictional provider of workplace equipment and support services. This handbook describes model rules for its office, warehouse and field-service teams. It applies to full-time, part-time and fixed-term employees. Visitors and contractors follow site safety, s


### 4. Semantic Chunking
Uses `SemanticChunker` with `sentence-transformers/all-MiniLM-L6-v2` embeddings to segment text based on semantic shifts between sentences rather than arbitrary character lengths.


In [4]:
# 1. Embedding model for semantic boundary detection and storage
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# 2. Semantic Chunker configured at 90th percentile threshold
semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90
)

semantic_chunks = semantic_chunker.split_documents(documents)

# 3. Enrich chunk metadata
for idx, chunk in enumerate(semantic_chunks):
    chunk.metadata["chunk_id"] = idx
    chunk.metadata["char_count"] = len(chunk.page_content)

print(f"Total Semantic Chunks generated: {len(semantic_chunks)}")
print("\n--- Sample Semantic Chunk #1 ---")
print(semantic_chunks[0].page_content[:260])


Total Semantic Chunks generated: 22

--- Sample Semantic Chunk #1 ---
Employee Rulebook Workplace policies for office, warehouse and field-service teams. FICTIONAL COMPANY Alderbridge Office Systems and all roles, procedures and policy terms in this handbook are fictional. This sample was created for academic use.


### 5. Local Persistent Vector Database (Chroma)
Indexes the semantic chunks into a local Chroma vector database:
- **Persistent Storage:** Stored in `./chroma_db` directory on disk.
- **Cosine Similarity:** Uses `"hnsw:space": "cosine"` for normalized distance queries.
- **Reproducible Management:** Dedicated `PersistentClient` with explicit collection naming.


In [5]:
db_dir = "./chroma_db"
collection_name = "alderbridge_employee_rulebook"

# Initialize local persistent client
client = chromadb.PersistentClient(path=db_dir)

# Cleanly recreate collection if it already exists
try:
    client.delete_collection(name=collection_name)
except Exception:
    pass

# Store and index in Chroma
vectorstore = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embeddings,
    client=client,
    collection_name=collection_name,
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"Successfully stored {len(semantic_chunks)} chunks in local Chroma DB at: '{db_dir}'")


Successfully stored 22 chunks in local Chroma DB at: './chroma_db'


### 6. Configurable Retrievers
LangChain vectorstores can be transformed into **Retrievers** (`vectorstore.as_retriever(...)`). Here we configure and compare two key retriever strategies:
1. **Standard Similarity Retriever:** Retrieves top-$k$ most similar chunks by cosine distance.
2. **Maximal Marginal Relevance (MMR) Retriever:** Balances topical relevance and diversity, penalizing redundant chunks (`lambda_mult=0.7`).


In [6]:
# 1. Standard Similarity Retriever
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 2. Maximal Marginal Relevance (MMR) Retriever (recommended for reducing redundancy)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.7
    }
)

print("Similarity Retriever and MMR Retriever are initialized and ready!")


Similarity Retriever and MMR Retriever are initialized and ready!


### 7. Querying with the Retriever
Use the retriever's `.invoke()` method to retrieve relevant context for domain questions, formatting the output with page citations and chunk IDs.


In [7]:
def format_docs(docs):
    """Formats retrieved documents into clean, cited context blocks."""
    formatted_blocks = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", 0) + 1
        chunk_id = doc.metadata.get("chunk_id", "N/A")
        block = f"[Source {i} | Page {page} | Chunk #{chunk_id}]:\n{doc.page_content.strip()}"
        formatted_blocks.append(block)
    return "\n\n".join(formatted_blocks)

test_queries = [
    "What are the standard office hours and break schedules?",
    "What is the policy for sick leave and absence notice?",
    "What are the rules regarding gifts and conflicts of interest?",
    "What is the procedure for resignation and returning company equipment?"
]

for query in test_queries:
    print("=" * 75)
    print(f"QUERY: {query}")
    print("=" * 75)
    retrieved_docs = mmr_retriever.invoke(query)
    print(format_docs(retrieved_docs))
    print()


QUERY: What are the standard office hours and break schedules?
[Source 1 | Page 4 | Chunk #7]:
### SECTION 02: Working time and pay

#### Policy 02.01: Schedules and breaks

Standard office hours are Monday through Friday, 9:00 a.m. to 5:30 p.m.

[Source 2 | Page 4 | Chunk #8]:
local time, with a 30-minute unpaid meal break. Site managers publish warehouse and field schedules at least seven calendar days before the schedule starts. Employees who cannot take a scheduled break tell their manager and record the actual time worked. #### Policy 02.02: Timekeeping and extra hours

Employees required to submit timesheets record actual start and finish times, unpaid breaks and assigned work outside scheduled hours. Submit the weekly timesheet by noon local time on the next business day after the workweek ends. Obtain manager approval before working extra hours whenever possible and record all extra hours, including unapproved hours. #### Policy 02.03: Attendance and lateness

Notify the manage

### 8. End-to-End LCEL RAG Chain
Integrates the retriever into a **LangChain Expression Language (LCEL)** pipeline:
```
retriever | format_docs ---> prompt_template ---> LLM ---> StrOutputParser
```
If a `GOOGLE_API_KEY` is present, it invokes `ChatGoogleGenerativeAI` (`gemini-1.5-flash`); otherwise, it synthesizes the retrieved prompt context cleanly.


In [8]:
prompt_template = ChatPromptTemplate.from_template("""You are an expert HR assistant answering questions based solely on the official Alderbridge Employee Rulebook.

Official Context:
{context}

Employee Question:
{question}

Answer with concise, policy-compliant instructions and cite the applicable Policy ID or Section:""")

# Build the LCEL Runnable Chain
rag_prompt_chain = (
    {"context": mmr_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
)

# Test the chain on a question
question = "What happens if I need to report a workplace injury or hazard?"
assembled_prompt = rag_prompt_chain.invoke(question)

print("=== Assembled Prompt Context via LCEL Chain ===")
print(assembled_prompt.to_string())

# Optional: Generate answer with Gemini LLM if GOOGLE_API_KEY is configured
if os.getenv("GOOGLE_API_KEY"):
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
        llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
        full_rag_chain = rag_prompt_chain | llm | StrOutputParser()
        answer = full_rag_chain.invoke(question)
        print("\n=== Gemini LLM Answer ===")
        print(answer)
    except Exception as e:
        print(f"\nLLM call error: {e}")
else:
    print("\n[Note: Set GOOGLE_API_KEY environment variable to generate direct LLM answers via Gemini.]")


=== Assembled Prompt Context via LCEL Chain ===
Human: You are an expert HR assistant answering questions based solely on the official Alderbridge Employee Rulebook.

Official Context:
[Source 1 | Page 7 | Chunk #16]:
### SECTION 05: Safety and information security

#### Policy 05.01: Site safety and emergency response

Complete assigned safety training before operating equipment or entering restricted work areas. Wear site-required protective equipment and stop work if conditions are unsafe. In an emergency, follow posted site instructions, contact site response or local emergency services as appropriate, and go to the designated assembly point when told to evacuate. #### Policy 05.02: Injuries, hazards and near misses

Arrange immediate assistance for injuries and notify the site lead as soon as it is safe. Report injuries, hazards and near misses in the incident register before the end of the shift where practicable. If the employee cannot submit the report, the shift lead records i

### 9. Interactive Chatbot: Ask Questions Directly
Use the `ask_chatbot()` function below to ask any custom question about the Alderbridge Employee Rulebook. You can change the question text and re-run the cell anytime!


In [10]:
def ask_chatbot(question: str):
    """
    Ask any question about the company rulebook.
    Retrieves the most relevant policy chunks and formats citations.
    """
    print(f"\n❓ Question: {question}\n" + "-" * 70)
    docs = mmr_retriever.invoke(question)
    
    print("📄 Retrieved Policy Context:")
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", 0) + 1
        chunk_id = doc.metadata.get("chunk_id", "N/A")
        print(f"\n[Source {i} | Page {page} | Chunk #{chunk_id}]:")
        print(doc.page_content.strip())
    print("\n" + "=" * 70)

# Try asking custom questions:
ask_chatbot("What are the standard office working hours?")



❓ Question: What are the standard office working hours?
----------------------------------------------------------------------
📄 Retrieved Policy Context:

[Source 1 | Page 4 | Chunk #7]:
### SECTION 02: Working time and pay

#### Policy 02.01: Schedules and breaks

Standard office hours are Monday through Friday, 9:00 a.m. to 5:30 p.m.

[Source 2 | Page 4 | Chunk #8]:
local time, with a 30-minute unpaid meal break. Site managers publish warehouse and field schedules at least seven calendar days before the schedule starts. Employees who cannot take a scheduled break tell their manager and record the actual time worked. #### Policy 02.02: Timekeeping and extra hours

Employees required to submit timesheets record actual start and finish times, unpaid breaks and assigned work outside scheduled hours. Submit the weekly timesheet by noon local time on the next business day after the workweek ends. Obtain manager approval before working extra hours whenever possible and record all extra ho